# Phase 4: Build release tables

This notebook makes the release-table work inspectable. It joins Phase 2 articles and selected species–PMID matches and the full PubMed PMID index (so `n_articles` is not limited to the fetched sample) with Phase 3 gate decisions and the 73 domain-question answers, then writes versioned-input Parquet tables to `data/04_tables/release/`.

The transformations live in `pathogen_atlas_v2/tables/` and the orchestration in `pathogen_atlas_v2/pipelines/04_build_tables.py`. The first-evidence and time-to-publication metrics reproduce Version 1's emergence table logic. Nothing here turns Jev probabilities into pathogen labels beyond the configured 0.75 decision filters. Set `run_phase: true` in `configs/notebook_parameters.yaml` to build the release.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pathogen_atlas_v2").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathogen_atlas_v2 import notebook_utils as nbutils
from pathogen_atlas_v2.common.io import resolve_path
NOTEBOOK_CONFIG_PATH = "configs/notebook_parameters.yaml"
PARAMS = nbutils.notebook_phase_parameters("phase4_table_creation", NOTEBOOK_CONFIG_PATH)

import importlib
phase4_module = importlib.import_module("pathogen_atlas_v2.pipelines.04_build_tables")
phase4_preflight = phase4_module.phase4_preflight
build_phase4_tables = phase4_module.build_phase4_tables


## Inputs and processing steps

1. Load Phase 2 abstracts, selected-for-fetch species–PMID pairs, and the unsampled canonical PMID index for full PubMed article counts.
2. Load the full Phase 3 decision ledger for the configured run and apply the configured disease-evidence threshold and join article title, abstract, and publication year.
3. Load the cached domain expert probabilities for the configured run and derive yes/no answers at the configured question threshold. Keep only pairs that passed the disease-evidence gate.
4. Aggregate the Version 1 style full-search `n_articles`, accepted-evidence `n_causation`, and first-evidence year per species, plus the 73 feature summaries.
5. Reproduce emergence metrics: publication target year, years from first evidence to target, and evidence papers in the initial publication-year window.
6. Write Parquet tables, CSV web exports, and table metadata.

In [ ]:
# Inspect whether Phase 1–3 releases needed by the joins are available.
preflight = phase4_preflight(PARAMS)
display(preflight)

# These are the functions used by the builder; each transformation is also
# independently callable with DataFrames for interactive analysis.
from pathogen_atlas_v2.tables.species_table import build_species_summary
from pathogen_atlas_v2.tables.feature_tables import build_feature_tables
from pathogen_atlas_v2.tables.emergence_tables import build_emergence_table
from pathogen_atlas_v2.tables.abstract_counts import build_abstract_count_table
from pathogen_atlas_v2.tables.web_exports import export_release_tables

display({
    "species_summary": build_species_summary.__doc__,
    "feature_tables": build_feature_tables.__doc__,
    "emergence_table": build_emergence_table.__doc__,
    "abstract_counts": build_abstract_count_table.__doc__,
    "web_exports": export_release_tables.__doc__,
})

In [ ]:
if PARAMS.get("run_phase", False):
    phase4_result = build_phase4_tables(PARAMS)
else:
    phase4_result = {"status": "preflight_only", "reason": "Set run_phase: true to build the Phase 4 tables."}
display(phase4_result)

In [ ]:
release_dir = resolve_path(PARAMS.get("release_dir", "data/04_tables/release"))
release_tables = {
    "evidence_records": release_dir / "evidence_records.parquet",
    "species_summary": release_dir / "species_summary.parquet",
    "domain_expert_answers": release_dir / "domain_expert_answers.parquet",
    "species_feature_summary": release_dir / "species_feature_summary.parquet",
    "species_feature_matrix": release_dir / "species_feature_matrix.parquet",
    "abstract_counts": release_dir / "abstract_counts.parquet",
    "emergence": release_dir / "emergence.parquet",
}
display(nbutils.existing_outputs(release_tables))
for name, path in release_tables.items():
    print(name)
    display(nbutils.read_table_if_exists(path, n_rows=5))